# 05b — Net Return Analysis

## Purpose
Notebook 05 evaluated the frozen 2014 approval policy using *principal loss*. However, judging a policy by principal loss alone ignores the interest earned on loans. This notebook applies the frozen PD boundary to the 2015 out-of-time cohort and evaluates it using undiscounted *net cash return*, leveraging the post-origination payment fields extracted in Notebook 01b.

## Methodology: Net-Cash Definition
In Notebook 01b, a reconciliation audit confirmed that the `total_pymnt` field already includes the `recoveries` amount for charged-off loans. To avoid double-counting recoveries, this notebook defines undiscounted net cash using the following formula:

`net_cash = total_pymnt - collection_recovery_fee - funded_amnt`

The net return on the portfolio is calculated as `net_cash / funded_amnt`. This is a cash-basis measure representing returned principal and collected interest minus collection fees. It is undiscounted and excludes internal funding and servicing costs.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import sys
from pathlib import Path
from scipy.sparse import issparse

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

from src.features.credit_features import (
    BASELINE_FEATURES,
    split_by_issue_month,
    make_row_features
)

# Load frozen policy from Notebook 05
policy_file = project_root / "models" / "checkpoints" / "pd_policy_2014_unweighted_isotonic.json"
with policy_file.open("r", encoding="utf-8") as stream:
    policy = json.load(stream)

optimal_threshold = policy["pd_threshold"]

# Load Data
pd_file = project_root / "data" / "processed" / "pd_cohort_2011_2015_36m.parquet"
payment_file = project_root / "data" / "processed" / "payment_outcomes_2011_2015_36m.parquet"

data = pd.read_parquet(pd_file)
payment_data = pd.read_parquet(payment_file)
payment_data["id"] = payment_data["id"].astype("string")

splits = split_by_issue_month(data)
test_ready = make_row_features(splits["test"])
test_ready["id"] = test_ready["id"].astype("string")

# Load preprocessor and model
preprocessor = joblib.load(project_root / "models" / "checkpoints" / policy["preprocessor_file"])
calibrated_model = joblib.load(project_root / "models" / "checkpoints" / policy["model_file"])

# Transform and Predict
X_test = preprocessor.transform(test_ready[BASELINE_FEATURES])
feature_names = preprocessor.get_feature_names_out()

if issparse(X_test):
    X_test_named = pd.DataFrame.sparse.from_spmatrix(X_test, index=test_ready.index, columns=feature_names)
else:
    X_test_named = pd.DataFrame(X_test, index=test_ready.index, columns=feature_names)

test_ready["PD"] = calibrated_model.predict_proba(X_test_named)[:, 1]
print(f"Loaded 2015 Test Cohort: {len(test_ready):,} loans")
print(f"Frozen PD Threshold: {optimal_threshold:.6f}")

Loaded 2015 Test Cohort: 283,026 loans
Frozen PD Threshold: 0.169611


In [2]:
# Merge payment outcomes
# Note: Notebook 01b confirmed that `total_pymnt` already includes `recoveries`.
test_portfolio = test_ready[["id", "PD", "funded_amnt"]].merge(
    payment_data[["id", "total_pymnt", "collection_recovery_fee"]],
    on="id",
    how="left",
    validate="one_to_one"
)

# Calculate undiscounted net cash
test_portfolio["net_cash"] = (
    test_portfolio["total_pymnt"] 
    - test_portfolio["collection_recovery_fee"] 
    - test_portfolio["funded_amnt"]
)

# Apply frozen policy
approved_2015 = test_portfolio[test_portfolio["PD"] <= optimal_threshold]
rejected_2015 = test_portfolio[test_portfolio["PD"] > optimal_threshold]

approved_net_cash = approved_2015["net_cash"].sum()
rejected_net_cash = rejected_2015["net_cash"].sum()
all_net_cash = test_portfolio["net_cash"].sum()

approved_roi = approved_net_cash / approved_2015["funded_amnt"].sum()
rejected_roi = rejected_net_cash / rejected_2015["funded_amnt"].sum()
all_roi = all_net_cash / test_portfolio["funded_amnt"].sum()

print("2015 OOT Evaluation — Net Cash Return")
print("-" * 45)
print(f"Approved loans ({len(approved_2015):,}): Net Cash = ${approved_net_cash:,.2f} | Return = {approved_roi:.2%}")
print(f"Rejected loans ({len(rejected_2015):,}): Net Cash = ${rejected_net_cash:,.2f} | Return = {rejected_roi:.2%}")
print(f"Accept-all baseline ({len(test_portfolio):,}): Net Cash = ${all_net_cash:,.2f} | Return = {all_roi:.2%}")

2015 OOT Evaluation — Net Cash Return
---------------------------------------------
Approved loans (195,051): Net Cash = $187,091,704.86 | Return = 7.18%
Rejected loans (87,975): Net Cash = $43,177,727.27 | Return = 4.24%
Accept-all baseline (283,026): Net Cash = $230,269,432.13 | Return = 6.35%


## Conclusion

While Notebook 05 demonstrated that the approved portfolio suffered a 5.27% principal loss rate, this notebook proves that the approved portfolio was ultimately highly profitable, generating over $187 million in undiscounted net cash for a 7.18% return.

Crucially, while the loans rejected by the model were still marginally profitable on average (yielding a 4.24% return), the model successfully concentrated capital into the highest-performing segments. By applying the model, the portfolio achieved a 7.18% return compared to the 6.35% Accept-All baseline. While this strict policy meant leaving approximately $43 million of absolute net cash on the table from rejected loans, it significantly improved the overall efficiency and risk-adjusted return of the deployed capital.